# Inspect PartEdit-Bench For Part-Level TDM Localization

This notebook checks whether PartEdit-Bench can support Harry Yang's requested pilot study: 10-15 cases balanced by target part size, with ground-truth masks for evaluating Follow-Your-Shape TDM localization.

## Goal

Before renting a GPU or running Follow-Your-Shape, confirm the dataset fields, image/mask formats, prompt structure, and mask-area distribution. The notebook should produce a small candidate table, not download or commit the full dataset into the repository.

## Setup

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import get_dataset_config_names, load_dataset
from PIL import Image


def find_repo_root(start):
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "core").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DATASET_ID = "Aleksandar/PartEdit-Bench"
OUTPUT_DIR = REPO_ROOT / "core" / "data" / "partedit_subset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREFERRED_SPLIT = "real"
IMAGE_FIELD = "original_image"
EDITED_REFERENCE_FIELD = "partedit"
MASK_FIELD = "gt_mask"
SOURCE_PROMPT_FIELD = "prompt_original"
TARGET_PROMPT_FIELD = "prompt_changed"
PART_FIELD = "part"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

print(f"Repository root: {REPO_ROOT}")
print(f"Subset output directory: {OUTPUT_DIR}")

## 1. Inspect Dataset Configs And Splits

In [ ]:
configs = get_dataset_config_names(DATASET_ID)
configs

In [ ]:
dataset_kwargs = {}
if configs:
    dataset_kwargs["name"] = configs[0]

dataset = load_dataset(DATASET_ID, **dataset_kwargs)
dataset

## 2. Inspect Actual Schema And Example Content

In [ ]:
split_name = PREFERRED_SPLIT if PREFERRED_SPLIT in dataset else list(dataset.keys())[0]
split = dataset[split_name]

print("split:", split_name)
print("num rows:", len(split))
split.features

In [ ]:
sample = split[0]

field_rows = []
for field_name, value in sample.items():
    if isinstance(value, Image.Image):
        preview = f"PIL.Image size={value.size} mode={value.mode}"
    else:
        preview = value
    field_rows.append({
        "field": field_name,
        "python_type": type(value).__name__,
        "preview": preview,
    })

pd.DataFrame(field_rows)

In [ ]:
task_fields = [
    "id",
    "class_name",
    "subject",
    "part",
    "edit",
    "seed",
    SOURCE_PROMPT_FIELD,
    TARGET_PROMPT_FIELD,
    "p2p_prompt",
    "p2p_template",
    "instructp2p_edit1",
    "instructp2p_edit2",
    "instructp2p_edit3",
]

pd.DataFrame([
    {"field": field, "value": sample[field]}
    for field in task_fields
    if field in sample
])

## 3. Set Follow-Your-Shape Field Mapping

PartEdit-Bench already provides the fields needed for this diagnostic. `prompt_original` and `prompt_changed` map directly to Follow-Your-Shape's source and target prompts, while `gt_mask` is used only for evaluation.

In [ ]:
field_mapping = pd.DataFrame([
    {"experiment_role": "source image", "dataset_field": IMAGE_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "source prompt", "dataset_field": SOURCE_PROMPT_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "target prompt", "dataset_field": TARGET_PROMPT_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "ground-truth part mask", "dataset_field": MASK_FIELD, "used_for": "localization evaluation only"},
    {"experiment_role": "target part label", "dataset_field": PART_FIELD, "used_for": "case description and grouping"},
    {"experiment_role": "PartEdit reference image", "dataset_field": EDITED_REFERENCE_FIELD, "used_for": "optional qualitative reference"},
])

field_mapping

In [ ]:
required_fields = [
    IMAGE_FIELD,
    MASK_FIELD,
    SOURCE_PROMPT_FIELD,
    TARGET_PROMPT_FIELD,
    PART_FIELD,
]
missing_fields = [field for field in required_fields if field not in sample]

if missing_fields:
    raise KeyError(f"Missing required fields: {missing_fields}. Available fields: {list(sample.keys())}")

print("Required PartEdit-Bench fields are available.")

## 4. Visual Check One Example

In [ ]:
def as_mask_array(value):
    if isinstance(value, Image.Image):
        arr = np.asarray(value.convert("L"))
    else:
        arr = np.asarray(value)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr > 0


def mask_area_ratio(mask_value):
    mask = as_mask_array(mask_value)
    return float(mask.mean())


image = sample[IMAGE_FIELD]
mask = as_mask_array(sample[MASK_FIELD])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("original_image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title(f"gt_mask ratio={mask.mean():.3f}")
axes[2].imshow(image)
axes[2].imshow(mask, alpha=0.35, cmap="Reds")
axes[2].set_title("mask overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 5. Compute Real-Split Mask-Area Distribution

The `real` split is the first candidate because it has 13 cases, which already falls inside Harry's requested 10-15 case range. This section checks whether those 13 real cases are balanced across target part sizes.

In [ ]:
def build_mask_table(dataset_dict, split_names):
    records = []
    for current_split_name in split_names:
        current_split = dataset_dict[current_split_name]
        for idx in range(len(current_split)):
            row = current_split[idx]
            records.append({
                "case_uid": f"{current_split_name}_{idx:04d}",
                "dataset_split": current_split_name,
                "dataset_index": idx,
                "id": row["id"],
                "class_name": row["class_name"],
                "subject": row["subject"],
                "part": row[PART_FIELD],
                "edit": row["edit"],
                "prompt_original": row[SOURCE_PROMPT_FIELD],
                "prompt_changed": row[TARGET_PROMPT_FIELD],
                "mask_area_ratio": mask_area_ratio(row[MASK_FIELD]),
            })
    return pd.DataFrame(records).sort_values("mask_area_ratio").reset_index(drop=True)


real_mask_table = build_mask_table(dataset, ["real"] if "real" in dataset else [split_name])
real_mask_table

In [ ]:
ax = real_mask_table["mask_area_ratio"].hist(bins=12, figsize=(8, 4))
ax.set_title("PartEdit-Bench real split target mask area ratios")
ax.set_xlabel("mask area / image area")
ax.set_ylabel("case count")

## 6. Compare Real And Synth Splits

Because the real split may not contain enough size diversity, also inspect the combined `real + synth` candidate pool. The final pilot subset can still prefer real images, but synth cases can fill missing part-size ranges.

In [ ]:
available_splits = [name for name in ["real", "synth"] if name in dataset]
all_mask_table = build_mask_table(dataset, available_splits)

all_mask_table.groupby("dataset_split").agg(
    num_cases=("id", "count"),
    min_mask_ratio=("mask_area_ratio", "min"),
    median_mask_ratio=("mask_area_ratio", "median"),
    max_mask_ratio=("mask_area_ratio", "max"),
).reset_index()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, current_split_name in zip(axes, available_splits):
    values = all_mask_table.loc[
        all_mask_table["dataset_split"] == current_split_name,
        "mask_area_ratio",
    ]
    values.hist(bins=12, ax=ax)
    ax.set_title(f"{current_split_name} mask area ratios")
    ax.set_xlabel("mask area / image area")
    ax.set_ylabel("case count")

plt.tight_layout()

## 7. Build Quantile-Based Part-Size Buckets

Fixed thresholds such as `<5%`, `5-15%`, and `>15%` produce too few large cases in this dataset. For Harry's request, use tertiles of ground-truth mask-area ratio so the selected cases are balanced across the dataset's available part-size range.

In [ ]:
bucketed_table = all_mask_table.copy()
bucketed_table["part_size"] = pd.qcut(
    bucketed_table["mask_area_ratio"].rank(method="first"),
    q=3,
    labels=["small", "medium", "large"],
)

bucket_summary = (
    bucketed_table.groupby(["dataset_split", "part_size"], observed=True)
    .size()
    .rename("num_cases")
    .reset_index()
)

bucket_summary

In [ ]:
bucket_ranges = (
    bucketed_table.groupby("part_size", observed=True)
    .agg(
        num_cases=("id", "count"),
        min_mask_ratio=("mask_area_ratio", "min"),
        median_mask_ratio=("mask_area_ratio", "median"),
        max_mask_ratio=("mask_area_ratio", "max"),
    )
    .reset_index()
)

bucket_ranges

## 8. Build A Candidate Balanced Subset

Select up to 5 cases per part-size bucket. Within each bucket, prefer `real` cases first and use `synth` cases only when needed for balance. This gives a 10-15 case candidate set while keeping Harry's part-size balance requirement explicit.

In [ ]:
selection_table = bucketed_table.copy()
selection_table["split_priority"] = selection_table["dataset_split"].map({"real": 0, "synth": 1}).fillna(2)

candidate_subset = (
    selection_table.sort_values(["part_size", "split_priority", "mask_area_ratio", "dataset_index"])
    .groupby("part_size", group_keys=False, observed=True)
    .head(5)
    .drop(columns=["split_priority"])
    .sort_values(["part_size", "mask_area_ratio"])
    .reset_index(drop=True)
)

candidate_subset[[
    "case_uid",
    "dataset_split",
    "dataset_index",
    "id",
    "part_size",
    "mask_area_ratio",
    "class_name",
    "subject",
    "part",
    "edit",
    "prompt_original",
    "prompt_changed",
]]

In [ ]:
candidate_subset.groupby(["part_size", "dataset_split"], observed=True).size().rename("num_cases").reset_index()

In [ ]:
preview_path = OUTPUT_DIR / "cases_preview.csv"
candidate_subset.to_csv(preview_path, index=False)
preview_path

## 9. Review Candidate Cases

The automatic subset is only a candidate. Use the following checks to inspect images, masks, prompts, and size buckets before writing the final `cases.json` manifest.

In [ ]:
review_columns = [
    "case_uid",
    "dataset_split",
    "dataset_index",
    "part_size",
    "mask_area_ratio",
    "class_name",
    "subject",
    "part",
    "edit",
    "prompt_original",
    "prompt_changed",
]

candidate_subset[review_columns]

In [ ]:
def resized_mask_for_image(mask_value, image):
    mask = as_mask_array(mask_value)
    image_width, image_height = image.size
    if mask.shape != (image_height, image_width):
        mask_image = Image.fromarray((mask.astype(np.uint8) * 255), mode="L")
        mask_image = mask_image.resize(image.size, resample=Image.Resampling.NEAREST)
        mask = np.asarray(mask_image) > 0
    return mask


def get_dataset_row(case_row):
    return dataset[case_row["dataset_split"]][int(case_row["dataset_index"])]


def show_candidate_case(case_row):
    dataset_row = get_dataset_row(case_row)
    image = dataset_row[IMAGE_FIELD]
    mask = resized_mask_for_image(dataset_row[MASK_FIELD], image)
    reference = dataset_row[EDITED_REFERENCE_FIELD]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    title = (
        f"{case_row['case_uid']} | {case_row['part_size']} | "
        f"ratio={case_row['mask_area_ratio']:.3f} | part={case_row['part']}"
    )
    fig.suptitle(title, fontsize=11)

    axes[0].imshow(image)
    axes[0].set_title("original_image")
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("gt_mask")
    axes[2].imshow(image)
    axes[2].imshow(mask, alpha=0.35, cmap="Reds")
    axes[2].set_title("mask overlay")
    axes[3].imshow(reference)
    axes[3].set_title("PartEdit reference")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("source prompt:", dataset_row[SOURCE_PROMPT_FIELD])
    print("target prompt:", dataset_row[TARGET_PROMPT_FIELD])
    print("edit:", dataset_row["edit"])


show_candidate_case(candidate_subset.iloc[0])

In [ ]:
for _, case_row in candidate_subset.iterrows():
    show_candidate_case(case_row)

## 10. Check Bucket Balance

This check verifies that the candidate subset has the intended number of cases per quantile-based part-size bucket and records the actual mask-area range behind each label.

In [ ]:
candidate_balance = (
    candidate_subset.groupby(["part_size", "dataset_split"], observed=True)
    .size()
    .rename("num_cases")
    .reset_index()
)

candidate_balance

In [ ]:
candidate_ranges = (
    candidate_subset.groupby("part_size", observed=True)
    .agg(
        num_cases=("case_uid", "count"),
        min_mask_ratio=("mask_area_ratio", "min"),
        median_mask_ratio=("mask_area_ratio", "median"),
        max_mask_ratio=("mask_area_ratio", "max"),
    )
    .reset_index()
)

candidate_ranges

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for part_size, frame in candidate_subset.groupby("part_size", observed=True):
    ax.scatter(
        [str(part_size)] * len(frame),
        frame["mask_area_ratio"],
        label=str(part_size),
        s=60,
    )
ax.set_title("Selected candidate mask-area ratios by part-size bucket")
ax.set_xlabel("part-size bucket")
ax.set_ylabel("mask area / image area")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

## 11. Write Reviewed Manifest

Edit `EXCLUDED_CASE_UIDS` after visual inspection. The manifest writes only metadata and dataset indices; it does not write image or mask files.

In [ ]:
EXCLUDED_CASE_UIDS = []

reviewed_subset = candidate_subset[~candidate_subset["case_uid"].isin(EXCLUDED_CASE_UIDS)].copy()

if not 10 <= len(reviewed_subset) <= 15:
    raise ValueError(f"Reviewed subset has {len(reviewed_subset)} cases; expected 10-15.")

reviewed_subset[review_columns]

In [ ]:
manifest_records = []
for row in reviewed_subset.to_dict(orient="records"):
    manifest_records.append({
        "case_uid": row["case_uid"],
        "dataset_id": DATASET_ID,
        "dataset_split": row["dataset_split"],
        "dataset_index": int(row["dataset_index"]),
        "dataset_row_id": int(row["id"]),
        "part_size": str(row["part_size"]),
        "part_size_method": "tertiles of gt_mask area ratio over real+synth PartEdit-Bench candidates",
        "mask_area_ratio": float(row["mask_area_ratio"]),
        "source_image_field": IMAGE_FIELD,
        "gt_mask_field": MASK_FIELD,
        "source_prompt": row["prompt_original"],
        "target_prompt": row["prompt_changed"],
        "part": row["part"],
        "subject": row["subject"],
        "class_name": row["class_name"],
        "edit": row["edit"],
    })

manifest = {
    "dataset_id": DATASET_ID,
    "selection_policy": "quantile-balanced by gt_mask area ratio; prefer real cases, fill with synth when needed",
    "num_cases": len(manifest_records),
    "cases": manifest_records,
}

manifest

In [ ]:
WRITE_REVIEWED_MANIFEST = False

manifest_path = OUTPUT_DIR / "cases.json"
if WRITE_REVIEWED_MANIFEST:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Wrote {manifest_path}")
else:
    print("Set WRITE_REVIEWED_MANIFEST = True after manual review to write cases.json")
    print(f"Target path: {manifest_path}")

## Next Step

After writing `core/data/partedit_subset/cases.json`, inspect the manifest diff and commit it. Downloaded images, masks, and generated model outputs should remain outside git.